# O Oráculo Secreto das Prililis
>Autor: Contreras, Abraham

Um teste de personalidade em que você **vira um prilili** por uma noite inteira, do pôr do sol à alvorada, e descobre qual das cinco prililis (Prilili, Lilika, Nili, Priki ou Kiki) vive em você.

**Como jogar:** execute as células de cima para baixo (ou *Run All*). A última célula inicia a experiência. Responda digitando o número da alternativa; `v` volta uma etapa e `s` encerra.

**Onde mexer:**
- `PAUSA_ECO`, `ARQUIVO_FICHA` e os pesos de pontuação: célula 1
- Personalidades, perguntas e alternativas: célula 2 (cada alternativa leva uma prilili principal, uma secundária e um eco)
- Cores, tipografia e animações: célula 3
- Regras de pontuação e fluxo: célula 4

*Os resultados são uma brincadeira inspirada nos prililis, não história oficial.*

In [ ]:
# O Oráculo Secreto das Prililis · célula 1: ajustes e utilidades
import hashlib
import html
import itertools
import random
import secrets
import time
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path

from IPython.display import HTML, clear_output, display

# ── Ajustes (pode mexer à vontade) ──────────────────────────────
PAUSA_ECO = 2.4                       # segundos que o eco de cada resposta fica na tela
ARQUIVO_FICHA = "ficha_prilili.html"  # onde a ficha é guardada (caminho relativo)
PESO_PRINCIPAL = 3                    # pontos para a prilili da alternativa
PESO_SECUNDARIO = 1                   # pontos para a prilili "vizinha" da alternativa

rng = random.SystemRandom()

# Nomes das cinco prililis do guia
PR, LI, NI, PK, KI = "Prilili", "Lilika", "Nili", "Priki", "Kiki"


class Saida(Exception):
    """Levantada quando a pessoa digita 's' para sair."""


Opcao = namedtuple("Opcao", "texto principal secundaria eco")


def o(texto, principal, secundaria, eco):
    """Cria uma alternativa: texto, prilili principal, prilili secundária e eco."""
    return Opcao(texto, principal, secundaria, eco)


def esc(texto):
    return html.escape(str(texto), quote=True)


def hex_para_rgb(cor):
    cor = cor.lstrip("#")
    return tuple(int(cor[i:i + 2], 16) for i in (0, 2, 4))


def rgb_para_hex(rgb):
    return "#%02x%02x%02x" % tuple(round(c) for c in rgb)


def percentuais(pontos):
    """Converte pontos em porcentagens inteiras que somam exatamente 100."""
    total = sum(pontos.values()) or 1
    brutos = {k: v * 100 / total for k, v in pontos.items()}
    base = {k: int(v) for k, v in brutos.items()}
    falta = 100 - sum(base.values())
    for k in sorted(brutos, key=lambda k: brutos[k] - base[k], reverse=True)[:falta]:
        base[k] += 1
    return base

In [ ]:
# O Oráculo Secreto das Prililis · célula 2: o universo (prililis, noite e perguntas)

FADAS = {
    PR: {
        "cor": "#c79bff", "cor2": "#ff8ad4",
        "titulo": "A Guardiã dos Sonhos",
        "descricao": ("Você encontra beleza onde quase ninguém olha. Com delicadeza, imaginação "
                      "e muita esperança, transforma pequenos momentos em grandes encantos."),
        "dom": "Fazer florescer aquilo que parecia impossível.",
        "toque": "um coração que cuida de todo mundo sem fazer alarde",
        "aparencia": "Asas lilás de brilho leve e um laço rosa na cabeça.",
        "fruta": "Amora colhida com orvalho",
        "hora": "O começo da noite, quando as flores abrem",
        "casa": "Chalé de cogumelo cercado de flores que brilham",
        "pedra": "Quartzo-rosa",
        "bando": "Lembra aniversários, guarda a melhor fatia e percebe quando alguém ficou quieto demais.",
        "cuidado": "Tem dificuldade de dizer não, e a agenda de favores vive lotada.",
        "dupla": "Kiki. Ela faz você rir até nos dias cinzentos; você dá a ela um lugar calmo para pousar.",
        "novela": "Priki. Entre idas e vindas, é a novela mais comentada da floresta.",
        "sussurro": "Milili prilili…",
        "traducao": "Tem uma flor esperando por você.",
    },
    LI: {
        "cor": "#e9d8b4", "cor2": "#9fd6a8",
        "titulo": "A Tecelã dos Mistérios",
        "descricao": ("Curiosidade e intuição são o seu radar: você adora descobrir o que existe por "
                      "trás das aparências, e sua imaginação transforma o desconhecido em aventura."),
        "dom": "Encontrar segredos escondidos entre as sombras.",
        "toque": "um olhar que enxerga os detalhes que todo mundo perdeu",
        "aparencia": "Pálida, redondinha e com o sorriso de quem sabe mais do que conta.",
        "fruta": "Jabuticaba (a casca escura guarda um segredo doce)",
        "hora": "A madrugada silenciosa",
        "casa": "Árvore oca com biblioteca e gavetas secretas",
        "pedra": "Obsidiana (diz que é só decoração)",
        "bando": "Guarda as informações: sabe de tudo, conta pouco e sempre tem um plano pronto.",
        "cuidado": "Planos elaborados demais, que de vez em quando dão errado... de novo.",
        "dupla": "Nili. Ela descobre os caminhos; você descobre o que estava escondido neles.",
        "novela": "Prilili. Seus planos contra ela raramente saem como o esperado.",
        "sussurro": "Ñilili prilili ñaca.",
        "traducao": "Eu sei de uma coisa que você ainda não sabe.",
    },
    NI: {
        "cor": "#8fa3bd", "cor2": "#c9d3e0",
        "titulo": "O Espírito Livre da Floresta",
        "descricao": ("Você segue o próprio caminho, valoriza sua liberdade e não tem medo de mudar "
                      "de direção. Sua presença traz movimento e vida para qualquer lugar."),
        "dom": "Descobrir caminhos onde antes só havia obstáculos.",
        "toque": "uma vontade deliciosa de mudar de direção na hora certa",
        "aparencia": "Pele cinza de sombra, cabelo escuro e uma cauda que balança ao andar.",
        "fruta": "Caju, colhido no pé, caminhando",
        "hora": "A hora em que o vento muda de lado",
        "casa": "Cabana pendurada em galhos que mudam de lugar",
        "pedra": "Opala, que muda de cor conforme o caminho",
        "bando": "Puxa o grupo para fora do caminho conhecido, e quase sempre o desvio vale a pena.",
        "cuidado": "Some sem avisar e depois pergunta por que ninguém sabia onde estava.",
        "dupla": "Priki. Ela corre, você desvia; juntos, chegam a lugares que nenhum mapa registra.",
        "novela": "Lilika. Ela quer mapa de tudo; você quer perder o mapa.",
        "sussurro": "Trilili, trilili!",
        "traducao": "Vamos por outro caminho.",
    },
    PK: {
        "cor": "#5cc7ff", "cor2": "#b9a6ff",
        "titulo": "A Centelha Indomável",
        "descricao": ("Coragem, espontaneidade e muita energia: você prefere experimentar a passar a "
                      "vida só imaginando, e encara desafios com criatividade e uma pitada de ousadia."),
        "dom": "Transformar desafios em oportunidades inesperadas.",
        "toque": "uma coragem que não espera ninguém pedir duas vezes",
        "aparencia": "Azul-lavanda, orelhas enormes e braços cruzados de quem já decidiu.",
        "fruta": "Manga, mordida sem cerimônia",
        "hora": "A hora do primeiro voo",
        "casa": "Torre de tronco no alto da colina ventosa",
        "pedra": "Citrino, cor de faísca",
        "bando": "Vai na frente quando todo mundo hesita e volta para buscar quem ficou para trás.",
        "cuidado": "Age primeiro e pensa depois (as idas e vindas de novela estão aí para provar).",
        "dupla": "Nili. Ela mostra atalhos que você jamais pensaria em testar.",
        "novela": "Prilili. Brigam, fazem as pazes, brigam de novo, e ninguém perde um capítulo.",
        "sussurro": "Prilili niaka!",
        "traducao": "Segura firme, eu vou na frente.",
    },
    KI: {
        "cor": "#ff6b5e", "cor2": "#ffc27a",
        "titulo": "A Estrela Brincalhona",
        "descricao": ("Você encontra diversão nas situações mais improváveis. Imprevisível e cheio de "
                      "espírito, sabe que até os dias mais estranhos podem esconder uma boa história."),
        "dom": "Espalhar alegria e surpreender até o destino.",
        "toque": "um humor que desarma até o clima mais pesado",
        "aparencia": "Asas coral que cintilam como brasas e uma pose de quem entra em cena.",
        "fruta": "Carambola (é uma estrela, claro)",
        "hora": "A hora da festa, seja ela qual for",
        "casa": "Toca colorida cheia de portinhas e surpresas",
        "pedra": "Turmalina-melancia, escolhida só por ser engraçada",
        "bando": "Salva a noite quando o clima pesa: uma careta, uma rima e todo mundo respira.",
        "cuidado": "Transforma tudo em piada, até quando alguém só queria um abraço silencioso.",
        "dupla": "Prilili. Ela transforma as suas piadas em flores.",
        "novela": "Lilika. Cada plano dela termina em risada, geralmente a sua.",
        "sussurro": "Hihihi trilili ñaka!",
        "traducao": "Aposto que consigo fazer você rir agora.",
    },
}

# Texto sobre o povo prilili, a partir do Guia Prilili
POVO = [
    "Os prililis são seres mágicos, brincalhões e pacíficos que vivem nas florestas.",
    "Alimentam-se principalmente de frutas e têm hábitos majoritariamente noturnos.",
    "Suas casas e edifícios vêm das estruturas da natureza, como fungos e árvores.",
    "O dinheiro deles é feito de pedras preciosas e de recursos valiosos para a floresta.",
    "Voam com as próprias asas e vivem por um bom tempo.",
]


def Q(cap, cena, pergunta, *opcoes):
    """Monta uma pergunta: capítulo, cena, enunciado e as 5 alternativas."""
    return {"cap": cap, "cena": cena, "pergunta": pergunta, "opcoes": list(opcoes)}


# A jornada é uma noite inteira, do pôr do sol à alvorada. Cada etapa (ato) tem
# 2 ou 3 perguntas possíveis e uma é sorteada, então cada noite é diferente.
ATOS = [
    {"hora": "Pôr do sol", "topo": "#9a3fe3", "base": "#2d1068", "perguntas": [
        Q("O primeiro ritual",
          "O sol se esconde atrás da copa e a sua casa de cogumelo acorda junto com você, espreguiçando o chapéu de musgo. Prililis vivem de noite: o dia, para você, termina agora.",
          "Antes de sair, qual é o seu primeiro ritual?",
          o("Cumprimentar as flores do jardim, uma por uma, para ver quais abriram hoje.", PR, KI, "As flores abrem um pouco mais só para você."),
          o("Reler o caderno de planos da noite, com setas, rasuras e um plano B secreto.", LI, PK, "O plano B sussurra: “eu sempre estive aqui”."),
          o("Sair pela janela, não pela porta, só para ver o que muda.", NI, KI, "O vento aprova a escolha da saída."),
          o("Esticar as asas e testar um voo rasante antes de acordar direito.", PK, NI, "Folhas giram atrás de você como confete."),
          o("Ensaiar uma entrada triunfal para ninguém em especial.", KI, PR, "A casa de cogumelo aplaude com o chapéu.")),
        Q("A casa que cresceu com você",
          "Prililis não constroem casas: elas brotam de fungos e árvores. A sua nasceu exatamente do jeito que você precisava.",
          "Como ela é por dentro? Qual é o seu esconderijo perfeito?",
          o("Um jardim secreto coberto de pétalas luminosas.", PR, KI, "As pétalas acendem uma a uma, como se esperassem você."),
          o("Uma biblioteca escondida dentro de uma árvore oca.", LI, PR, "Um livro se afasta da estante para abrir lugar."),
          o("Uma cabana suspensa entre galhos que se movem.", NI, PK, "A cabana balança de leve: ela também gosta de mudar."),
          o("Uma torre no alto de um penhasco ventoso.", PK, LI, "Lá do alto, a floresta parece um mapa à sua espera."),
          o("Uma toca colorida cheia de passagens e surpresas.", KI, NI, "Uma passagem secreta solta uma risadinha.")),
    ]},
    {"hora": "Primeiras estrelas", "topo": "#7a33d6", "base": "#230d58", "perguntas": [
        Q("A ceia de frutas",
          "Prililis vivem de frutas, e a noite começa pela cesta. Hoje há uma fruta que você nunca viu: brilha, cheira a festa e está na ponta de um galho fino.",
          "Como você chega até ela?",
          o("Pede com carinho ao galho, que sempre se curva para quem trata bem.", PR, KI, "O galho se inclina e a fruta cai na sua mão."),
          o("Observa quem já tentou e por que falhou, antes de tentar.", LI, PK, "Você descobre que o galho é mais esperto que a fruta."),
          o("Esquece essa fruta e segue o cheiro de outra, que vem de mais longe.", NI, LI, "O cheiro leva você aonde ninguém olhou."),
          o("Voa direto até a ponta do galho, sem pensar duas vezes.", PK, KI, "O galho estala, mas segura. Você ri do susto."),
          o("Propõe uma corrida: quem pegar primeiro divide com todo mundo.", KI, PR, "A ceia vira corrida, e ninguém reclama.")),
        Q("A última fatia",
          "Uma fruta enorme, madura, e três prililis famintas olhando para ela com o olhar mais convincente possível.",
          "O que você faz?",
          o("Divide em partes iguais e guarda a melhor para quem chegou por último.", PR, KI, "Três sorrisos e uma fatia a mais: conta fechada."),
          o("Descobre quem está com mais fome de verdade antes de decidir.", LI, PR, "Você lê os olhares como quem lê um mapa."),
          o("Pega a sua parte e vai comer em outro lugar, sem cerimônia.", NI, KI, "A liberdade tem gosto doce e um pouco pegajoso."),
          o("Corta ao meio com uma asa e entrega sem discurso.", PK, NI, "Um corte limpo e nenhum drama. Hoje, pelo menos."),
          o("Inventa um jogo: quem fizer a melhor careta fica com o miolo.", KI, PR, "A careta vencedora entra para a história da floresta.")),
    ]},
    {"hora": "Noite alta", "topo": "#5b2bc4", "base": "#190a45", "perguntas": [
        Q("A trilha desconhecida",
          "Entre as árvores, uma trilha que não estava ali ontem.",
          "O que você faz?",
          o("Segue uma luz delicada que parece chamar por você.", PR, KI, "A luz respira mais devagar, como se agradecesse."),
          o("Examina os símbolos estranhos gravados nos troncos.", LI, NI, "Os símbolos formam uma frase. Você a decora."),
          o("Abandona a trilha e inventa o próprio caminho.", NI, KI, "Onde você pisa, o musgo já faz curva."),
          o("Corre para descobrir o que existe do outro lado.", PK, LI, "O outro lado ainda não sabe o que vem aí."),
          o("Dá um nome absurdo à trilha e começa a explorá-la cantando.", KI, PK, "A trilha agora se chama “Ruela do Sorvete Torto”. Ela adorou.")),
        Q("A porta na árvore",
          "No tronco mais velho da floresta, uma portinha acabou de aparecer. Tem maçaneta e tudo.",
          "Você...",
          o("Decora a entrada com flores e espera alguém especial aparecer.", PR, LI, "A porta parece bem mais simpática de enfeite."),
          o("Tenta descobrir quem construiu a porta e por quê.", LI, PK, "Há marcas de ferramenta: alguém passou por aqui há pouco."),
          o("Abre a porta e segue sem planejar o destino.", NI, PK, "A porta dá para um lugar que você não saberia pedir."),
          o("Bate três vezes e entra sem hesitar.", PK, KI, "A terceira batida abre mais do que a porta."),
          o("Pergunta se a porta tem senha e tenta adivinhar cantando.", KI, LI, "A porta resmunga, mas a melodia quase funcionou.")),
    ]},
    {"hora": "Hora das corujas", "topo": "#4325a8", "base": "#13093a", "perguntas": [
        Q("Uma pequena criatura perdida",
          "Algo minúsculo e tremelicante se esconde atrás de um cogumelo: uma criatura mágica que perdeu o caminho de casa.",
          "Como você reage?",
          o("Oferece carinho e promete ajudá-la a voltar para casa.", PR, KI, "A criatura se aninha na sua mão: confiança entregue."),
          o("Pergunta de onde ela veio e que magia conhece.", LI, PR, "Ela responde com um brilho que você nunca tinha visto."),
          o("Convida a criatura para explorar a floresta com você.", NI, PR, "Dois passos depois, vocês já se perderam juntos. Que ótimo."),
          o("Organiza uma missão de resgate imediatamente.", PK, NI, "A missão tem nome, plano e bandeira feita de folha."),
          o("Faz uma careta para ela rir antes de pensar no plano.", KI, PR, "O medo vira soluço de riso: o plano pode esperar.")),
        Q("Um dragão do tamanho de um pardal",
          "Um dragão minúsculo pousa no seu ombro, fareja o seu cabelo e solta uma fumacinha com cheiro de canela.",
          "O que você diz?",
          o("“Você parece precisar de um abraço.”", PR, LI, "O dragão apoia a cabecinha no seu pescoço e ronrona fumaça."),
          o("“Interessante... de qual espécie você é?”", LI, KI, "Ele estufa o peito: ninguém nunca tinha perguntado."),
          o("“Vamos ver até onde conseguimos voar!”", NI, PK, "Ele abre as asinhas. A noite fica menor."),
          o("“Você sabe cuspir fogo? Precisamos testar isso!”", PK, NI, "Uma chama do tamanho de um grão de pipoca. Aplausos."),
          o("“Ótimo, agora tenho um aquecedor portátil.”", KI, NI, "O dragão finge ofensa, mas não sai do ombro.")),
    ]},
    {"hora": "Lua cheia", "topo": "#34198f", "base": "#0f0731", "perguntas": [
        Q("Uma pedra entre as raízes",
          "Entre as raízes, algo brilha: uma pedra preciosa do tamanho de uma noz. No mercado das copas, ela rende moedas para a estação inteira.",
          "O que você faz com ela?",
          o("Troca por sementes e planta um jardim para todo mundo.", PR, KI, "O mercado inteiro vai florescer em algumas semanas."),
          o("Guarda no bolso e investiga por que estava tão fácil de achar.", LI, NI, "Pedras raras não costumam ficar paradas. Esta ficou."),
          o("Troca por uma viagem sem destino e sem passagem de volta.", NI, KI, "O vento já conhece o seu novo endereço: nenhum."),
          o("Compra equipamento novo e testa em um voo ainda hoje.", PK, KI, "O primeiro teste termina em um looping que ninguém esquece."),
          o("Joga para o alto: se cair, ótimo; se virar estrela, melhor ainda.", KI, PR, "A pedra sobe, pisca uma vez e... decide cair.")),
        Q("O presente da árvore encantada",
          "Uma árvore muito antiga abre uma fresta no tronco e oferece um único presente.",
          "Qual você escolhe?",
          o("Uma flor que nunca perde as pétalas.", PR, KI, "Ela exala o perfume da primeira primavera."),
          o("Uma chave capaz de abrir portas invisíveis.", LI, NI, "Em algum lugar, uma porta invisível sentiu cócegas."),
          o("Um mapa que muda conforme os seus desejos.", NI, LI, "O mapa já está mudando. Você nem decidiu ainda."),
          o("Uma pedra que libera energia quando você precisa.", PK, PR, "A pedra esquenta na palma, como um coração teimoso."),
          o("Um cogumelo que conta uma piada diferente a cada dia.", KI, NI, "A primeira piada é péssima. Você gargalha mesmo assim.")),
    ]},
    {"hora": "Meia-noite", "topo": "#2d1580", "base": "#0b0627", "perguntas": [
        Q("O segredo no galho",
          "Em um galho próximo, duas fadinhas cochicham. Uma delas aponta para você, sussurra algo no ouvido da outra, e as duas começam a rir sem parar.",
          "Como você reage?",
          o("Sorri de volta sem perguntar: o riso delas é bonito demais para interromper.", PR, KI, "O riso delas se mistura ao seu, e a noite fica mais leve."),
          o("Se aproxima devagar para ouvir a segunda rodada do segredo.", LI, PK, "A segunda rodada é mais baixa. Elas são boas nisso."),
          o("Dá de ombros e segue caminho: se for importante, vai chegar até você.", NI, LI, "Atrás de você, o riso vira um eco que não pede resposta."),
          o("Voa até elas e pergunta de frente: “é sobre mim?”", PK, LI, "Um segundo de silêncio. Depois, risada em dobro."),
          o("Começa a rir também, sem saber do quê. A piada é a risada.", KI, PR, "Ninguém sabe do que está rindo. É perfeito.")),
        Q("A poção que desandou",
          "Você misturava uma poção simples quando ela borbulhou, mudou de cor duas vezes e deu errado.",
          "O que acontece depois?",
          o("Ela faz nascer flores em todos os lugares.", PR, KI, "Até a colher de pau resolve brotar."),
          o("Ela revela uma mensagem que estava invisível.", LI, NI, "A mensagem diz apenas: “continue”."),
          o("Ela transforma a sala numa floresta selvagem.", NI, PK, "Cipós entram pela janela como visitas bem-vindas."),
          o("Ela explode em faíscas e você tenta controlar tudo.", PK, LI, "Você prende uma faísca com o chapéu. Vitória parcial."),
          o("Ela faz todo mundo falar rimando por uma hora.", KI, NI, "Rima de ar, rima de mar... ninguém consegue parar.")),
        Q("Quando ninguém está olhando",
          "A noite está no auge, a clareira ficou vazia e, por alguns minutos, você é só você.",
          "O que você costuma fazer?",
          o("Imaginar histórias bonitas sobre as pessoas ao redor.", PR, LI, "Na sua história, todo mundo tem um final gentil."),
          o("Observar detalhes que todo mundo parece ignorar.", LI, PR, "Detalhe do dia: uma folha com sete nervuras, uma delas torta."),
          o("Sonhar com lugares distantes e novas possibilidades.", NI, PK, "No sonho, o horizonte é uma porta entreaberta."),
          o("Inventar projetos e começar algo na mesma hora.", PK, KI, "O projeto já tem nome, e você nem tem material."),
          o("Fazer coisas aleatórias só para ver o que acontece.", KI, NI, "Resultado: um cogumelo de chapéu virado. Cientificamente interessante.")),
    ]},
    {"hora": "Madrugada", "topo": "#3b1f9b", "base": "#16094a", "perguntas": [
        Q("A lua desaparece",
          "De repente, a lua some do céu e a floresta prende a respiração.",
          "Qual é a sua reação?",
          o("Acende pequenas flores para iluminar a floresta.", PR, KI, "As flores acendem uma a uma, e a escuridão perde a pressa."),
          o("Investiga os sinais do céu para descobrir o motivo.", LI, NI, "Há uma sombra de nuvem... que não é nuvem."),
          o("Aproveita para conhecer lugares que nunca visitou.", NI, LI, "No escuro, até o caminho antigo parece novo."),
          o("Reúne as criaturas e parte para resolver o problema.", PK, PR, "Vinte lanternas de vaga-lume seguem atrás de você."),
          o("Organiza uma festa noturna para ninguém ficar triste.", KI, PR, "A festa da lua ausente tem até bolo de lua minguante.")),
        Q("O sapo e a coroa torta",
          "Um sapo solene sobe numa vitória-régia, ajeita uma coroa torta e declara que é o rei legítimo da floresta. Ninguém lembra de ter coroado ninguém.",
          "Qual é a sua posição?",
          o("Oferece um chá e escuta a história dele: todo mundo merece ser ouvido.", PR, LI, "O sapo funga, comovido. A coroa escorrega, mas o chá aquece."),
          o("Investiga de onde veio a coroa: tem alguma coisa que não fecha.", LI, PK, "Na parte de dentro da coroa há um nome raspado."),
          o("Diz “claro, majestade” e segue viagem: reis vêm e vão.", NI, KI, "O sapo adora ser levado a sério. Você adora ir embora."),
          o("Desafia o sapo para um duelo de saltos: quem vencer fica com a coroa.", PK, KI, "O sapo salta longe, mas você voa. Regras são regras."),
          o("Faz uma reverência exagerada e pede o primeiro decreto: “segunda-feira vira feriado”.", KI, PR, "O decreto é aprovado por aclamação das rãs.")),
    ]},
    {"hora": "Alvorada", "topo": "#1aa3c9", "base": "#6a2fc0", "perguntas": [
        Q("O que importa na amizade",
          "O céu começa a clarear. Cansados e inteiros, vocês dividem as últimas frutas da noite.",
          "O que mais importa em uma amizade?",
          o("Carinho, confiança e pequenos gestos sinceros.", PR, LI, "A última fruta vai para quem mais precisava."),
          o("Conversas profundas e respeito pelos segredos.", LI, PR, "Alguns silêncios são a melhor conversa."),
          o("Liberdade para ser quem você realmente é.", NI, KI, "Ninguém pediu que você fosse outro. Nem hoje."),
          o("Lealdade, coragem e estar presente nas dificuldades.", PK, PR, "Ao seu lado, alguém ajeita as asas e fica."),
          o("Rir junto até quando tudo parece dar errado.", KI, NI, "A risada vira o último brinde da noite.")),
        Q("O desejo da floresta",
          "A floresta, satisfeita com a sua noite, oferece um único desejo.",
          "Qual é?",
          o("Que ninguém perca a capacidade de sonhar.", PR, KI, "Em algum canto do mundo, alguém acorda sorrindo."),
          o("Conhecer os segredos mais antigos do universo.", LI, NI, "A floresta ri baixinho: “um de cada vez”."),
          o("Poder explorar o mundo sem nunca perder a liberdade.", NI, PK, "Uma brisa leva o seu desejo para além do horizonte."),
          o("Ter coragem para enfrentar qualquer desafio.", PK, PR, "Algo quente acende no peito, como uma brasa boa."),
          o("Que sempre exista motivo para rir, mesmo nos dias difíceis.", KI, PR, "A floresta inteira solta uma risadinha concordando.")),
        Q("A única magia",
          "Antes de dormir, a floresta ensina uma magia só e deixa você escolher qual.",
          "Qual você domina?",
          o("Curar a tristeza e fazer jardins renascerem.", PR, KI, "Onde você toca, algo pequeno e verde recomeça."),
          o("Descobrir verdades escondidas no tempo.", LI, PR, "O tempo abre uma gaveta e nem finge surpresa."),
          o("Viajar para qualquer lugar sem deixar rastros.", NI, LI, "Você some e reaparece, só para o vento perceber."),
          o("Controlar raios de luz e explosões de energia.", PK, KI, "Uma faísca azul dança nos seus dedos. Cuidado com as sobrancelhas."),
          o("Transformar qualquer desastre numa aventura divertida.", KI, NI, "O desastre da semana passada agora tem trilha sonora.")),
    ]},
]

# Perguntas de desempate: aparecem só se duas ou mais prililis terminarem empatadas.
# Cada alternativa tem uma única prilili (sem secundária).
DESEMPATE = [
    Q("A última pergunta",
      "O oráculo hesita entre duas luzes (ou mais). Só você pode desempatar.",
      "Quando a noite acaba, o que você leva no bolso?",
      o("Uma pétala, para lembrar que o delicado também resiste.", PR, None, "A pétala murmura: “resiste, sim”."),
      o("Um bilhetinho com uma pergunta que ficou sem resposta.", LI, None, "A pergunta é boa. A resposta virá."),
      o("Nada, de propósito: bolso vazio tem espaço para o caminho.", NI, None, "O bolso vazio parece mais leve que o céu."),
      o("Uma faísca guardada para o primeiro desafio do dia seguinte.", PK, None, "A faísca esquenta o bolso. Ela também está ansiosa."),
      o("Uma piada nova, ainda quente.", KI, None, "Você tenta segurar o riso. Não consegue.")),
    Q("A marca na floresta",
      "O oráculo hesita. Uma pergunta só, a mais sincera de todas.",
      "Se pudesse deixar uma marca na floresta, qual seria?",
      o("Uma clareira onde qualquer um possa descansar.", PR, None, "As árvores abrem espaço sem fazer barulho."),
      o("Um símbolo que só os curiosos vão entender.", LI, None, "Os curiosos já estão a caminho."),
      o("Uma trilha nova, ainda sem nome.", NI, None, "A trilha começa a existir sob os seus pés."),
      o("Uma ponte que ninguém achava possível.", PK, None, "A ponte range, mas atravessa."),
      o("Uma risada que ecoe de árvore em árvore.", KI, None, "A floresta devolve a risada com juros.")),
    Q("A despedida da coruja",
      "A coruja da alvorada se despede de você.",
      "Qual é a sua última frase para ela?",
      o("“A noite foi linda por causa de você.”", PR, None, "A coruja pisca devagar: recado recebido."),
      o("“Fica entre nós o que a gente viu hoje.”", LI, None, "A coruja jura silêncio. Corujas são boas nisso."),
      o("“Amanhã eu passo por outro caminho.”", NI, None, "A coruja sorri: ela já sabia."),
      o("“Na próxima, eu voo mais alto!”", PK, None, "A coruja ajeita as penas, desafiada."),
      o("“Guarda uma risada para mim até a noite que vem.”", KI, None, "A coruja guarda. Até deu um pio de prévia.")),
]


def validar_dados():
    """Garante que perguntas e alternativas estão consistentes antes de jogar."""
    def checar(q, com_secundaria):
        principais = sorted(op.principal for op in q["opcoes"])
        assert principais == sorted(FADAS), f"{q['cap']}: cada prilili deve aparecer uma vez"
        for op in q["opcoes"]:
            assert op.texto and op.eco, f"{q['cap']}: alternativa incompleta"
            if com_secundaria:
                assert op.secundaria in FADAS and op.secundaria != op.principal, \
                    f"{q['cap']}: secundária inválida em “{op.texto}”"

    for ato in ATOS:
        for q in ato["perguntas"]:
            checar(q, com_secundaria=True)
    for q in DESEMPATE:
        checar(q, com_secundaria=False)


validar_dados()

In [ ]:
# O Oráculo Secreto das Prililis · célula 3: identidade visual
# Paleta tirada da capa do Guia Prilili: violeta noturno, magenta, ciano e o cobre da letra manuscrita.

CSS = """<style>
@import url('https://fonts.googleapis.com/css2?family=Pinyon+Script&family=Quicksand:wght@500;600;700&display=swap');
.pri-palco{--topo:#8f3be0;--base:#2b0f63;--papel:#f8f2ff;--suave:#d3c6f5;--cobre:#e0a869;--cobre2:#f7d6a6;--fc:#c79bff;
 --script:'Pinyon Script','Snell Roundhand','Apple Chancery','Lucida Handwriting','Brush Script MT',cursive;
 position:relative;overflow:hidden;box-sizing:border-box;max-width:760px;margin:6px auto;
 padding:clamp(18px,4vw,30px) clamp(16px,4.5vw,40px) 120px;border-radius:22px;
 color:var(--papel);font:500 16px/1.6 'Quicksand','Nunito','Segoe UI',system-ui,sans-serif;text-align:left;
 background:radial-gradient(110% 55% at 50% 104%,rgba(226,63,210,.32),transparent 62%),linear-gradient(180deg,var(--topo),var(--base));
 animation:pri-entra .5s ease backwards}
.pri-palco *{box-sizing:border-box}
.pri-palco::before{content:"";position:absolute;inset:0;pointer-events:none;opacity:.8;
 background-image:radial-gradient(1.6px 1.6px at 8% 12%,#fff,transparent),radial-gradient(1.2px 1.2px at 22% 30%,#fff,transparent),
 radial-gradient(1.8px 1.8px at 37% 8%,#fff,transparent),radial-gradient(1.2px 1.2px at 51% 22%,#fff,transparent),
 radial-gradient(1.6px 1.6px at 66% 11%,#fff,transparent),radial-gradient(1.2px 1.2px at 79% 27%,#fff,transparent),
 radial-gradient(1.8px 1.8px at 91% 9%,#fff,transparent),radial-gradient(1.2px 1.2px at 15% 47%,#fff,transparent),
 radial-gradient(1.4px 1.4px at 88% 52%,#fff,transparent),radial-gradient(1.2px 1.2px at 5% 70%,#fff,transparent),
 radial-gradient(1.4px 1.4px at 95% 76%,#fff,transparent);animation:pri-cintila 8s ease-in-out infinite alternate}
.pri-palco>*{position:relative;z-index:1}
.pri-palco>.pri-arvores{position:absolute;z-index:0;left:0;right:0;bottom:0;height:100px;pointer-events:none}
.pri-palco>.pri-flash{position:absolute;inset:0;z-index:5;background:#fff;opacity:0;pointer-events:none;animation:pri-flash 1.2s ease-out backwards}

.pri-topo{display:flex;align-items:center;gap:12px;margin-bottom:12px}
.pri-quem{display:flex;align-items:center;gap:10px;flex:1;font-family:var(--script);font-size:1.7rem;line-height:1;color:var(--cobre2)}
.pri-hora{display:flex;flex-direction:column;align-items:flex-end;font-weight:700;line-height:1.25}
.pri-hora span{font-weight:500;font-size:.85rem;color:var(--suave)}
.pri-aura{position:relative;width:42px;height:42px;flex:none;animation:pri-pulso 3.6s ease-in-out infinite}
.pri-aura i{position:absolute;inset:0;border-radius:50%;background:radial-gradient(circle at 35% 30%,rgba(255,255,255,.95),var(--c) 58%);box-shadow:0 0 24px 4px var(--c)}
.pri-aura i.novo{animation:pri-surge 1.4s ease backwards}
.pri-trilha{display:flex;gap:6px;margin:0 0 24px}
.pri-trilha span{flex:1;height:5px;border-radius:3px;background:rgba(255,255,255,.18)}
.pri-trilha span.feita{background:var(--c)}
.pri-trilha span.atual{background:var(--cobre2)}

.pri-cap{font-family:var(--script);font-size:clamp(2.1rem,6vw,2.9rem);line-height:1.1;color:var(--cobre2);margin:0 0 8px}
.pri-cena{color:var(--suave);max-width:62ch;margin:0 0 16px}
.pri-pergunta{font-weight:700;font-size:1.2rem;line-height:1.4;max-width:60ch;margin:0 0 14px}
.pri-opcao{display:flex;gap:14px;align-items:flex-start;padding:12px 16px;margin:9px 0;border:1px solid rgba(247,214,166,.3);border-radius:14px;background:rgba(14,6,44,.4)}
.pri-opcao.escolhida{border-color:var(--cobre2);background:rgba(247,214,166,.16)}
.pri-num{flex:none;width:28px;height:28px;border-radius:50%;display:grid;place-items:center;font-weight:700;font-size:.95rem;color:#2b0f63;background:var(--cobre2)}
.pri-eco{margin-top:20px;font-size:1.2rem;font-style:italic;color:var(--cobre2);max-width:56ch}
.pri-dica{margin-top:16px;font-size:.88rem;color:var(--suave)}
.pri-texto{max-width:60ch;margin:0 auto 14px;text-align:center}

.pri-asas-topo{display:flex;justify-content:center;margin:10px 0 0}
.pri-marca-caixa{animation:pri-foco 1.6s .3s ease backwards}
.pri-marca{font-family:var(--script);font-size:clamp(68px,17vw,124px);line-height:1.15;text-align:center;
 background:linear-gradient(180deg,#fbe3bd,#d89a5b 55%,#9a5a2a);-webkit-background-clip:text;background-clip:text;color:transparent;
 filter:drop-shadow(0 4px 18px rgba(255,180,110,.35))}
@supports not (background-clip:text){.pri-marca{color:var(--cobre2)}}
.pri-subtitulo{text-align:center;font-weight:700;font-size:1.25rem;letter-spacing:.02em;margin:0 0 6px}
.pri-mantra{text-align:center;font-style:italic;font-size:1.15rem;color:var(--cobre2);margin:14px 0 2px}
.pri-nota{text-align:center;font-size:.85rem;color:var(--suave);margin:0 0 22px}

.pri-linha-lenta{max-width:58ch;margin:0 0 14px;font-size:1.1rem}
.pri-linha-lenta b{color:var(--cobre2);font-weight:700}
.pri-pergaminho{margin:24px 0 8px;padding:6px 0 6px 18px;border-left:3px solid var(--cobre)}
.pri-pergaminho div{margin:6px 0;color:var(--suave);max-width:58ch}
.pri-pergaminho .pri-rotulo-pergaminho{color:var(--cobre2);font-weight:700}

.pri-orbes{display:flex;justify-content:center;align-items:center;gap:clamp(14px,5vw,40px);height:130px;margin:24px 0 10px}
.pri-orbe{width:clamp(30px,8vw,50px);aspect-ratio:1;border-radius:50%;background:radial-gradient(circle at 35% 30%,#fff,var(--c) 58%);box-shadow:0 0 26px 6px var(--c);animation:pri-pulso 2.4s ease-in-out infinite}
.pri-orbe.vence{animation:pri-vence 3.4s 2.8s ease forwards}
.pri-orbe.some{animation:pri-some 2.4s 2.8s ease forwards}

.pri-heroi{display:flex;justify-content:center;margin:8px 0 0;filter:drop-shadow(0 0 22px var(--fc))}
.pri-resultado{font-family:var(--script);text-align:center;line-height:1.1;font-size:clamp(52px,12vw,88px);
 background:linear-gradient(180deg,#fbe3bd,#d89a5b 60%,#9a5a2a);-webkit-background-clip:text;background-clip:text;color:transparent;padding:4px 0}
@supports not (background-clip:text){.pri-resultado{color:var(--cobre2)}}
.pri-titulo{text-align:center;font-size:1.25rem;font-weight:700;color:var(--fc);margin:0 0 4px}
.pri-nomefada{text-align:center;color:var(--suave);font-size:.95rem;margin:0 0 20px}
.pri-descricao{max-width:58ch;margin:0 auto 12px;text-align:center;font-size:1.08rem}
.pri-dom{max-width:58ch;margin:0 auto 20px;text-align:center;color:var(--cobre2);font-weight:700}
.pri-mapa-titulo{font-weight:700;margin:26px 0 6px}
.pri-barra{display:grid;grid-template-columns:78px 1fr 42px;align-items:center;gap:10px;margin:7px 0;font-size:.95rem}
.pri-barra.topo .pri-n{font-weight:700;color:var(--cobre2)}
.pri-trilho{height:10px;border-radius:5px;background:rgba(255,255,255,.15);overflow:hidden}
.pri-trilho i{display:block;height:100%;border-radius:5px;animation:pri-cresce 1.3s .3s ease backwards}
.pri-p{text-align:right;color:var(--suave)}
.pri-mistura{margin:16px 0 0;color:var(--suave)}
.pri-mistura b{color:var(--papel)}
.pri-ficha{margin-top:22px;border-top:1px solid rgba(247,214,166,.3)}
.pri-reg{display:grid;grid-template-columns:150px 1fr;gap:12px;padding:10px 0;border-bottom:1px solid rgba(247,214,166,.18)}
.pri-rotulo{color:var(--suave);font-size:.92rem}
.pri-sussurro{margin:24px 0 4px;padding-left:16px;border-left:3px solid var(--fc)}
.pri-lingo{font-style:italic;font-size:1.3rem}
.pri-trad{color:var(--suave);font-size:.9rem}
.pri-rodape{text-align:center;margin-top:26px}

.pri-menu{margin-top:30px}
.pri-menu .pri-opcao{padding:9px 14px;margin:7px 0}
.pri-aviso{margin:14px 0 0;color:var(--cobre2);font-weight:700}
.pri-galeria-item{display:flex;align-items:center;gap:16px;padding:10px 16px;margin:9px 0;border:1px solid rgba(247,214,166,.3);border-radius:14px;background:rgba(14,6,44,.4)}
.pri-galeria-item .pri-gn{font-family:var(--script);font-size:1.7rem;line-height:1;color:var(--cobre2)}
.pri-galeria-item .pri-gt{color:var(--suave);font-size:.92rem}
.pri-galeria-item .pri-num{margin-left:auto}

@keyframes pri-entra{from{opacity:0}to{opacity:1}}
@keyframes pri-surge{from{opacity:0}to{opacity:1}}
@keyframes pri-cintila{from{opacity:.45}to{opacity:.9}}
@keyframes pri-pulso{0%,100%{transform:scale(1)}50%{transform:scale(1.07)}}
@keyframes pri-foco{from{opacity:0;filter:blur(16px)}to{opacity:1;filter:blur(0)}}
@keyframes pri-flash{0%{opacity:0}3%{opacity:.95}10%{opacity:.15}16%{opacity:.8}100%{opacity:0}}
@keyframes pri-cresce{from{width:0}}
@keyframes pri-vence{from{transform:scale(1)}to{transform:scale(1.9);box-shadow:0 0 60px 18px var(--c)}}
@keyframes pri-some{to{opacity:.15;transform:scale(.7)}}
.pri-surge{animation:pri-surge .9s ease backwards}
@media (max-width:520px){.pri-reg{grid-template-columns:1fr;gap:2px}}
@media (prefers-reduced-motion:reduce){.pri-palco,.pri-palco *,.pri-palco::before{animation:none!important}.pri-palco>.pri-flash{display:none}}
</style>"""

_ids = itertools.count(1)


def asas(cor, cor2, px=64):
    """Asinhas em SVG, usadas como emblema de cada prilili."""
    uid = f"pri-g{next(_ids)}"
    asa = ('<path d="M60 48 C44 8 10 0 6 26 C3 44 30 56 60 52 Z"/>'
           '<path d="M60 56 C42 60 16 70 24 84 C34 94 54 76 60 60 Z"/>'
           '<path d="M58 50 C44 36 28 26 14 22" fill="none" stroke="#fff" stroke-opacity=".4"/>')
    return (f'<svg aria-hidden="true" width="{px}" height="{round(px * .75)}" viewBox="0 0 120 90" xmlns="http://www.w3.org/2000/svg">'
            f'<defs><linearGradient id="{uid}" x1="0" y1="0" x2="1" y2="1">'
            f'<stop offset="0" stop-color="{cor}"/><stop offset="1" stop-color="{cor2}"/></linearGradient></defs>'
            f'<g fill="url(#{uid})" fill-opacity=".88" stroke="#fff" stroke-opacity=".55" stroke-width="1">{asa}'
            f'<g transform="translate(120 0) scale(-1 1)">{asa}</g></g>'
            f'<ellipse cx="60" cy="56" rx="3" ry="15" fill="#fff" fill-opacity=".85"/></svg>')


def _arvores():
    fundo = [60, 88, 70, 96, 64, 84, 74, 92, 62, 86, 68, 94, 66, 82, 72, 90]
    frente = [46, 70, 52, 80, 56, 74, 48, 84, 58, 72, 50, 78, 54, 82, 60, 68, 48, 76]

    def faixa(alturas, larg):
        passo = 800 / (len(alturas) - 1)
        return " ".join(f"M{k * passo - larg:.0f} 100 L{k * passo:.0f} {100 - h} L{k * passo + larg:.0f} 100 Z"
                        for k, h in enumerate(alturas))

    return ('<div class="pri-arvores" aria-hidden="true"><svg width="100%" height="100%" viewBox="0 0 800 100" preserveAspectRatio="none">'
            f'<path d="{faixa(fundo, 34)}" fill="#1a0a45" fill-opacity=".75"/>'
            f'<path d="{faixa(frente, 28)}" fill="#09021f" fill-opacity=".95"/></svg></div>')


ARVORES = _arvores()


def palco(corpo, ato=0):
    """Envolve o conteúdo em um 'palco' com o céu da hora da noite correspondente."""
    topo, base = ATOS[ato]["topo"], ATOS[ato]["base"]
    return f'<div class="pri-palco" style="--topo:{topo};--base:{base}">{ARVORES}{corpo}</div>'


def mostrar(corpo, ato=0):
    """Troca a cena atual (limpa a tela e desenha a nova)."""
    clear_output(wait=True)
    display(HTML(CSS + palco(corpo, ato)))


def aura(de, para):
    return (f'<div class="pri-aura" aria-hidden="true"><i style="--c:{de}"></i>'
            f'<i class="novo" style="--c:{para}"></i></div>')


def cabecalho(jogador, ato, de, para):
    return ('<div class="pri-topo">'
            f'<div class="pri-quem">{asas("#f7d6a6", "#e0a869", 36)}<span>{esc(jogador.nome_fada)}</span></div>'
            f'<div class="pri-hora">{esc(ATOS[ato]["hora"])}<span>etapa {ato + 1} de {len(ATOS)}</span></div>'
            f'{aura(de, para)}</div>')


def trilha(concluidas, atual, cor):
    segs = "".join(
        f'<span class="{"feita" if i < concluidas else ("atual" if i == atual else "")}"></span>'
        for i in range(len(ATOS)))
    return f'<div class="pri-trilha" style="--c:{cor}">{segs}</div>'


def corpo_pergunta(jogador, ato, q, opcoes, cor_aura_atual, pode_voltar):
    itens = "".join(f'<div class="pri-opcao"><span class="pri-num">{n}</span><span>{esc(op.texto)}</span></div>'
                    for n, op in enumerate(opcoes, 1))
    volta = "v volta uma etapa, " if pode_voltar else ""
    return (cabecalho(jogador, ato, cor_aura_atual, cor_aura_atual) + trilha(ato, ato, cor_aura_atual)
            + f'<div class="pri-cap">{esc(q["cap"])}</div>'
            + f'<div class="pri-cena">{esc(q["cena"])}</div>'
            + f'<div class="pri-pergunta">{esc(q["pergunta"])}</div>'
            + itens
            + f'<div class="pri-dica">Digite o número da alternativa. {volta}s encerra.</div>')


def corpo_eco(jogador, ato, q, escolhida, aura_de, aura_para):
    return (cabecalho(jogador, ato, aura_de, aura_para) + trilha(ato + 1, -1, aura_para)
            + f'<div class="pri-cap">{esc(q["cap"])}</div>'
            + f'<div class="pri-opcao escolhida"><span class="pri-num">✓</span><span>{esc(escolhida.texto)}</span></div>'
            + f'<div class="pri-eco">{esc(escolhida.eco)}</div>')


def corpo_abertura():
    return ('<div class="pri-flash" aria-hidden="true"></div>'
            f'<div class="pri-asas-topo pri-surge" style="animation-delay:.9s">{asas("#f6a6e8", "#7fd8ff", 74)}</div>'
            '<div class="pri-marca-caixa"><div class="pri-marca">Prilili</div></div>'
            '<div class="pri-subtitulo pri-surge" style="animation-delay:1.6s">O oráculo secreto das prililis</div>'
            '<div class="pri-mantra pri-surge" style="animation-delay:2.1s">Piri katiki ña ña me</div>'
            '<div class="pri-nota pri-surge" style="animation-delay:2.1s">a frase que abre a floresta</div>'
            '<div class="pri-texto pri-surge" style="animation-delay:2.6s">Dizem que, muito além das trilhas conhecidas, existe uma floresta '
            'que escolhe quem pode conhecê-la. Hoje, ela escolheu você. E nesta noite você não será visita: será um prilili.</div>'
            '<div class="pri-texto pri-surge" style="animation-delay:3.1s">Não tente adivinhar a resposta certa. A magia reconhece o que é espontâneo.</div>')


def _linhas_lentas(textos, passo=1.3, inicio=.3):
    return "".join(f'<div class="pri-linha-lenta pri-surge" style="animation-delay:{inicio + i * passo:.1f}s">{t}</div>'
                   for i, t in enumerate(textos))


def corpo_prologo(jogador):
    textos = [
        "Você fecha os olhos e sente um formigamento nas costas.",
        "Duas asas translúcidas se abrem, leves como orvalho. De repente, a floresta fica enorme (ou você ficou pequeno o bastante para enxergá-la de verdade).",
        "Suas orelhas ficam pontudas, atentas a cada folha que se mexe.",
        f"Na língua da floresta, o seu nome é <b>{esc(jogador.nome_fada)}</b>.",
        "Você é prilili. A noite acaba de começar, e prililis pertencem à noite.",
    ]
    povo = "".join(f"<div>{esc(t)}</div>" for t in POVO[:3])
    return (f'<div class="pri-asas-topo pri-surge">{asas("#f6a6e8", "#7fd8ff", 70)}</div>'
            + _linhas_lentas(textos)
            + f'<div class="pri-pergaminho pri-surge" style="animation-delay:{.3 + len(textos) * 1.3:.1f}s">'
              f'<div class="pri-rotulo-pergaminho">O que todo prilili sabe</div>{povo}</div>')


def corpo_nova_noite(jogador):
    textos = ["Outra noite começa.", f"As asas de <b>{esc(jogador.nome_fada)}</b> se abrem de novo, e a floresta já tem perguntas novas."]
    return f'<div class="pri-asas-topo pri-surge">{asas("#f6a6e8", "#7fd8ff", 70)}</div>' + _linhas_lentas(textos, 1.2)


def corpo_orbes(vencedora):
    orbes = "".join(f'<span class="pri-orbe {"vence" if f == vencedora else "some"}" style="--c:{FADAS[f]["cor"]}"></span>' for f in FADAS)
    textos = ["A floresta fica em silêncio.", "Cinco pequenas luzes surgem entre as árvores.", "Uma delas começa a brilhar mais forte..."]
    return '<div class="pri-cap" style="text-align:center">O oráculo despertou</div>' + f'<div class="pri-orbes">{orbes}</div>' \
        + "".join(f'<div class="pri-linha-lenta pri-surge" style="animation-delay:{.4 + i * 1.2:.1f}s;text-align:center;margin-left:auto;margin-right:auto">{t}</div>'
                  for i, t in enumerate(textos))


def corpo_ficha(chave, jogador=None, resultado=None):
    f = FADAS[chave]
    pessoal = resultado is not None
    h = f'<div style="--fc:{f["cor"]}">'
    h += f'<div class="pri-heroi">{asas(f["cor"], f["cor2"], 130)}</div>'
    h += f'<div class="pri-resultado">{"Você é " if pessoal else ""}{esc(chave)}</div>'
    h += f'<div class="pri-titulo">{esc(f["titulo"])}</div>'
    if pessoal:
        humano = f" (para os humanos, {esc(jogador.nome)})" if jogador.nome else ""
        h += f'<div class="pri-nomefada">Na floresta, você atende por {esc(jogador.nome_fada)}{humano}</div>'
    else:
        h += '<div class="pri-nomefada">Uma das cinco prililis que conhecemos</div>'
    h += f'<div class="pri-descricao">{esc(f["descricao"])}</div>'
    h += f'<div class="pri-dom">{"Seu dom mágico" if pessoal else "Dom mágico"}: {esc(f["dom"])}</div>'
    if pessoal:
        pct = percentuais(resultado.pontos)
        h += '<div class="pri-mapa-titulo">O mapa das suas essências</div>'
        for nome in sorted(FADAS, key=lambda n: (-pct[n], n)):
            topo = " topo" if nome == chave else ""
            h += (f'<div class="pri-barra{topo}"><span class="pri-n">{esc(nome)}</span>'
                  f'<div class="pri-trilho"><i style="width:{pct[nome]}%;background:{FADAS[nome]["cor"]}"></i></div>'
                  f'<span class="pri-p">{pct[nome]}%</span></div>')
        comp = resultado.complementar
        h += (f'<div class="pri-mistura"><b>Seu toque de {esc(comp)}:</b> {esc(FADAS[comp]["toque"])}.</div>')
    linhas = [("Aparência", "aparencia"), ("Fruta preferida", "fruta"), ("Hora favorita", "hora"), ("Casa", "casa"),
              ("Pedra de estimação", "pedra"), ("No bando", "bando"), ("Ponto fraco", "cuidado"),
              ("Dupla ideal", "dupla"), ("Novela", "novela")]
    h += '<div class="pri-ficha">' + "".join(
        f'<div class="pri-reg"><span class="pri-rotulo">{rot}</span><span>{esc(f[k])}</span></div>' for rot, k in linhas) + '</div>'
    h += (f'<div class="pri-sussurro"><div class="pri-lingo">“{esc(f["sussurro"])}”</div>'
          f'<div class="pri-trad">Tradução livre: {esc(f["traducao"])} (nenhuma fada confirmou).</div></div>')
    if pessoal:
        h += '<div class="pri-rodape pri-mantra">Piri katiki ña ña me</div>'
    return h + '</div>'


def corpo_menu(aviso=""):
    itens = [("1", "Conhecer as outras prililis"), ("2", "O povo prilili"), ("3", "Guardar minha ficha em um arquivo"),
             ("4", "Viver outra noite"), ("0", "Encerrar")]
    lista = "".join(f'<div class="pri-opcao"><span class="pri-num">{n}</span><span>{t}</span></div>' for n, t in itens)
    extra = f'<div class="pri-aviso">{esc(aviso)}</div>' if aviso else ""
    return f'<div class="pri-menu"><div class="pri-pergunta">E agora?</div>{lista}{extra}</div>'


def corpo_galeria():
    itens = "".join(
        f'<div class="pri-galeria-item">{asas(f["cor"], f["cor2"], 52)}'
        f'<div><div class="pri-gn">{esc(n)}</div><div class="pri-gt">{esc(f["titulo"])}</div></div>'
        f'<span class="pri-num">{i}</span></div>' for i, (n, f) in enumerate(FADAS.items(), 1))
    return ('<div class="pri-cap">As cinco prililis</div><div class="pri-cena">Escolha uma para conhecer melhor.</div>'
            + itens + '<div class="pri-dica">Digite o número, ou 0 para voltar.</div>')


def corpo_povo():
    itens = "".join(f"<div>{esc(t)}</div>" for t in POVO)
    return (f'<div class="pri-asas-topo">{asas("#f6a6e8", "#7fd8ff", 70)}</div><div class="pri-cap" style="text-align:center">O povo prilili</div>'
            f'<div class="pri-pergaminho">{itens}</div>'
            '<div class="pri-dica">As personalidades deste oráculo são uma brincadeira inspirada nos prililis, não história oficial.</div>')


def corpo_despedida():
    return (f'<div class="pri-asas-topo">{asas("#f6a6e8", "#7fd8ff", 70)}</div>'
            '<div class="pri-texto" style="margin-top:18px">A floresta respeita a sua partida.<br>As prililis guardam o seu segredo até o seu retorno.</div>'
            '<div class="pri-mantra">Piri katiki ña ña me</div>')


def salvar_ficha(corpo):
    """Guarda a ficha em um HTML independente (caminho relativo ao notebook)."""
    pagina = ('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
              '<meta name="viewport" content="width=device-width, initial-scale=1">'
              '<title>Minha ficha de prilili</title></head>'
              '<body style="margin:0;padding:20px;background:#0c0522">' + CSS + palco(corpo, len(ATOS) - 1) + '</body></html>')
    caminho = Path(ARQUIVO_FICHA)
    caminho.write_text(pagina, encoding="utf-8")
    return caminho

In [ ]:
# O Oráculo Secreto das Prililis · célula 4: lógica e fluxo da experiência

@dataclass
class Jogador:
    nome: str        # como os humanos chamam a pessoa (pode ser vazio)
    nome_fada: str   # nome de prilili, gerado a partir do nome


@dataclass
class Resultado:
    vencedora: str
    complementar: str
    pontos: dict


SILABAS = ["pri", "li", "ki", "ni", "bli", "pi", "ka", "ti", "mi", "lo", "ña", "tu"]
FINAIS = ["li", "ki", "ka", "ni", "lu"]


def nome_de_fada(nome):
    """Gera um nome de prilili. O mesmo nome humano sempre dá o mesmo nome de fada."""
    base = (nome or secrets.token_hex(4)).casefold()
    reservados = {n.lower() for n in FADAS} | {"blili"}
    tentativa = 0
    while True:
        d = hashlib.sha256(f"{base}|{tentativa}".encode("utf-8")).digest()
        a, b = SILABAS[d[0] % len(SILABAS)], SILABAS[d[1] % len(SILABAS)]
        if a == b:
            b = SILABAS[(d[1] + 1) % len(SILABAS)]
        candidato = (a + b + FINAIS[d[2] % len(FINAIS)]).capitalize()
        if candidato.lower() not in reservados:
            return candidato
        tentativa += 1


def pontuar(respostas):
    """Soma os pontos (principal e secundário) e conta quantas vezes cada prilili foi a principal."""
    pontos = {f: 0 for f in FADAS}
    principais = {f: 0 for f in FADAS}
    for op in respostas:
        pontos[op.principal] += PESO_PRINCIPAL
        principais[op.principal] += 1
        if op.secundaria:
            pontos[op.secundaria] += PESO_SECUNDARIO
    return pontos, principais


def cor_aura(pontos):
    """Mistura as cores das prililis conforme os pontos. A aura revela a tendência sem entregar o resultado."""
    pesos = {f: p ** 2 for f, p in pontos.items()}
    total = sum(pesos.values())
    if not total:
        return "#d3c6f5"
    rgb = [sum(hex_para_rgb(FADAS[f]["cor"])[c] * w for f, w in pesos.items()) / total for c in range(3)]
    return rgb_para_hex(rgb)


def ler(validos, prompt="Sua escolha"):
    """Lê uma opção válida. 's' encerra a qualquer momento."""
    lista = "/".join(sorted(validos))
    texto = f"\n  {prompt} ({lista}): "
    while True:
        resposta = input(texto).strip().lower()
        if resposta in ("s", "sair"):
            raise Saida
        if resposta in validos:
            return resposta
        texto = f"\n  A floresta não entendeu. Digite {lista}: "


def pausar(mensagem="Pressione ENTER para continuar"):
    if input(f"\n  {mensagem}... ").strip().lower() in ("s", "sair"):
        raise Saida


def abertura():
    mostrar(corpo_abertura(), 0)
    nome = " ".join(input("\n  Como os humanos chamam você? (ENTER para entrar sem nome): ").split())[:24]
    return Jogador(nome=nome, nome_fada=nome_de_fada(nome))


def prologo(jogador, primeira_vez=True):
    mostrar(corpo_prologo(jogador) if primeira_vez else corpo_nova_noite(jogador), 0)
    pausar("Pressione ENTER para começar a noite" if primeira_vez else "Pressione ENTER para voar")


def jornada(jogador):
    """Percorre a noite: uma pergunta sorteada por etapa. 'v' volta uma etapa."""
    perguntas = [rng.choice(ato["perguntas"]) for ato in ATOS]
    respostas = []
    i = 0
    while i < len(perguntas):
        q = perguntas[i]
        opcoes = rng.sample(q["opcoes"], len(q["opcoes"]))
        pontos, _ = pontuar(respostas)
        aura_antes = cor_aura(pontos)
        mostrar(corpo_pergunta(jogador, i, q, opcoes, aura_antes, i > 0), i)
        validos = {"1", "2", "3", "4", "5"} | ({"v"} if i > 0 else set())
        resposta = ler(validos, "Qual caminho chama você?")
        if resposta == "v":
            respostas.pop()
            i -= 1
            continue
        escolhida = opcoes[int(resposta) - 1]
        respostas.append(escolhida)
        aura_depois = cor_aura(pontuar(respostas)[0])
        mostrar(corpo_eco(jogador, i, q, escolhida, aura_antes, aura_depois), i)
        time.sleep(PAUSA_ECO)
        i += 1
    return respostas


def desempatar(jogador, empatadas, pontos):
    """Pergunta final, só com as alternativas das prililis empatadas."""
    q = rng.choice(DESEMPATE)
    opcoes = [op for op in rng.sample(q["opcoes"], len(q["opcoes"])) if op.principal in empatadas]
    ultimo = len(ATOS) - 1
    cor = cor_aura(pontos)
    mostrar(corpo_pergunta(jogador, ultimo, {**q, "opcoes": opcoes}, opcoes, cor, False), ultimo)
    resposta = ler({str(n) for n in range(1, len(opcoes) + 1)}, "Qual luz você segue?")
    escolhida = opcoes[int(resposta) - 1]
    mostrar(corpo_eco(jogador, ultimo, q, escolhida, cor, cor), ultimo)
    time.sleep(PAUSA_ECO)
    return escolhida.principal


def apurar(respostas, jogador):
    """Decide a prilili: mais pontos, depois mais respostas principais, depois a pergunta de desempate."""
    pontos, principais = pontuar(respostas)
    chave = lambda f: (pontos[f], principais[f])
    melhor = max(chave(f) for f in FADAS)
    empatadas = [f for f in FADAS if chave(f) == melhor]
    if len(empatadas) > 1:
        vencedora = desempatar(jogador, empatadas, pontos)
        pontos[vencedora] += PESO_PRINCIPAL
    else:
        vencedora = empatadas[0]
    resto = [f for f in FADAS if f != vencedora]
    segundo = max(chave(f) for f in resto)
    complementar = rng.choice([f for f in resto if chave(f) == segundo])
    return Resultado(vencedora, complementar, pontos)


def revelar(resultado):
    mostrar(corpo_orbes(resultado.vencedora), len(ATOS) - 1)
    pausar("Pressione ENTER para se aproximar da luz")


def pos_resultado(jogador, resultado):
    """Menu final. Devolve True para viver outra noite e False para encerrar."""
    ultimo = len(ATOS) - 1
    ficha = corpo_ficha(resultado.vencedora, jogador, resultado)
    aviso = ""
    while True:
        mostrar(ficha + corpo_menu(aviso), ultimo)
        aviso = ""
        escolha = ler({"0", "1", "2", "3", "4"}, "O que deseja fazer agora?")
        if escolha == "0":
            return False
        if escolha == "4":
            return True
        if escolha == "1":
            while True:
                mostrar(corpo_galeria(), ultimo)
                n = ler({"0", "1", "2", "3", "4", "5"}, "Qual prilili quer conhecer?")
                if n == "0":
                    break
                mostrar(corpo_ficha(list(FADAS)[int(n) - 1]), ultimo)
                pausar("Pressione ENTER para voltar")
        elif escolha == "2":
            mostrar(corpo_povo(), ultimo)
            pausar("Pressione ENTER para voltar")
        elif escolha == "3":
            try:
                aviso = f"Ficha guardada em {salvar_ficha(ficha)}"
            except OSError as erro:
                aviso = f"Não consegui guardar o arquivo: {erro}"


def iniciar():
    """Executa a experiência completa."""
    try:
        jogador = abertura()
        primeira = True
        while True:
            prologo(jogador, primeira)
            primeira = False
            respostas = jornada(jogador)
            resultado = apurar(respostas, jogador)
            revelar(resultado)
            if not pos_resultado(jogador, resultado):
                break
        mostrar(corpo_despedida(), len(ATOS) - 1)
    except (KeyboardInterrupt, EOFError, Saida):
        mostrar(corpo_despedida(), len(ATOS) - 1)

In [ ]:
iniciar()